# LightningPoseTrack on Colab

Two modes. **Mode A (direct)** is the simple reliable path. **Mode B (Docker)**
works on some Colab runtimes but not all — it's experimental.

**Runtime**: Python 3, **GPU** accelerator (T4 / V100 / A100)

In [ ]:
# Choose mode: "direct" (default) or "docker"
MODE = "direct"

DOCKER_IMAGE = "kaarthikbalakrishnan/lightningposetrack:latest"
GITHUB_REPO = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"
REPO_DIR = "/content/LightningPoseTrack"

## Step 1: Mount Drive & Clone Repo

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

if not os.path.exists(REPO_DIR):
    !git clone -b {GIT_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print(f"Repo at {REPO_DIR}")

## Step 2: Verify GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Mode A: Direct Install (Recommended)

Installs packages directly. This is what every notebook (01-08) does in its first
pip install cell — this just gets it done early so you can skip those cells.

In [ ]:
if MODE == "direct":
    print("Installing system packages...")
    !apt-get update -qq && apt-get install -y -qq ffmpeg libgl1 libglib2.0-0 tesseract-ocr
    print("Installing Python packages...")
    !pip install -q -r requirements-colab.txt
    !pip install -q lightning-pose[all] omegaconf
    !pip install -q scikit-learn xgboost joblib
    print("Done. All dependencies installed.")

---
## Mode B: Docker Container (Experimental on Colab)

Requires a Colab VM that supports nested Docker. Most do not. If this fails,
switch MODE to "direct" at the top and re-run.

In [ ]:
import subprocess, time

docker_ok = False

if MODE != "docker":
    print("MODE is set to \"direct\". Skip this section.")
else:
    # Install Docker
    print("Installing docker.io...")
    !apt-get update -qq && apt-get install -y -qq docker.io

    # Start dockerd in background (Colab has no systemd)
    !nohup dockerd > /tmp/dockerd.log 2>&1 &
    time.sleep(5)

    # Check if it started
    r = subprocess.run(["docker", "info"], capture_output=True, text=True, timeout=10)
    if r.returncode == 0:
        docker_ok = True
        print("Docker daemon is running.")
    else:
        print(f"Docker failed: {r.stderr.strip()}")
        print("Colab does not support Docker on this runtime.")
        print("Set MODE = \"direct\" and re-run from Step 1.")

In [ ]:
if docker_ok:
    !docker pull {DOCKER_IMAGE}

In [ ]:
if docker_ok:
    !docker rm -f lightningposetrack 2>/dev/null || true
    !docker run -d --name lightningposetrack \
        --gpus all \
        -p 8888:8888 \
        -v {REPO_DIR}:/workspace \
        -v /content/drive:/content/drive \
        -w /workspace \
        {DOCKER_IMAGE} \
        jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root \
            --NotebookApp.token='' --NotebookApp.password='' \
            --NotebookApp.allow_origin='*'
    time.sleep(3)
    !docker logs lightningposetrack 2>&1 | tail -3

In [ ]:
if docker_ok:
    !docker ps --filter name=lightningposetrack
    print("---")
    !docker exec lightningposetrack python -c "
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
"

## Next Steps

**Mode A (direct):** Open the notebook you want from the `notebooks/` folder.
Each notebook handles its own pip installs, so you can skip those cells.

**Mode B (Docker):** Attach VS Code via **Dev Containers: Attach to Running
Container** → select `lightningposetrack`. Or `docker exec -it lightningposetrack bash`
in a terminal.

---

**Shortcut for Mode A users:** If you ran the direct install above, open notebook
03 and skip the first 6 pip install / clone cells.